In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU chưa được kích hoạt trên Colab!")

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
# ==================================================================================
# CELL MOUNT DRIVE DÙNG CHUNG — giữ NGUYÊN VĂN ở mọi notebook experiment
# (Baseline, Label Flipping, Backdoor, Defense). Chỉ sửa khối CẤU HÌNH bên dưới,
# và phải sửa giống hệt nhau ở mọi notebook.
# ==================================================================================
import os
import sys
import json    # dùng ở các cell sau
import csv     # dùng ở các cell sau
import shutil
import subprocess
from pathlib import Path

# ------------------------------ CẤU HÌNH (cả nhóm dùng chung) ---------------------
SHARED_FOLDER_NAME = 'data-poisoning-shared'   # tên thư mục gốc trong Drive (mọi người đặt đúng tên này)
REPO_URL = 'https://github.com/qtrong0205/data-poisoning-research'                                  # URL GitHub của project; để trống = lấy code từ <thư mục gốc>/repo
REPO_BRANCH = 'main'
LOCAL_PROJECT = Path('/content/data-poisoning-plantvillage')   # nơi chạy code (đĩa local của Colab, nhanh)
DRIVE_MOUNT = '/content/drive'
PIPELINES = ['baseline', 'label_flipping', 'backdoor', 'defenses']
# -----------------------------------------------------------------------------------

# 1. Mount Drive -----------------------------------------------------------------------
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT)
elif not Path(DRIVE_MOUNT).is_dir():
    raise RuntimeError('Cell này dành cho Google Colab (hoặc đặt DRIVE_MOUNT trỏ tới thư mục Drive).')

# 2. Tìm thư mục gốc dùng chung ----------------------------------------------------------
_candidates = []
if os.environ.get('DP_DRIVE_ROOT_OVERRIDE'):          # chỉ dùng khi muốn ép đường dẫn thủ công
    _candidates.append(Path(os.environ['DP_DRIVE_ROOT_OVERRIDE']))
_candidates.append(Path(DRIVE_MOUNT) / 'MyDrive' / SHARED_FOLDER_NAME)
_candidates.append(Path(DRIVE_MOUNT) / 'Shareddrives' / SHARED_FOLDER_NAME)
_candidates.extend(sorted(Path(DRIVE_MOUNT, 'Shareddrives').glob(f'*/{SHARED_FOLDER_NAME}')))
DRIVE_ROOT = next((c for c in _candidates if c.is_dir()), None)
if DRIVE_ROOT is None:
    raise FileNotFoundError(
        f'Không tìm thấy thư mục dùng chung "{SHARED_FOLDER_NAME}". Đã tìm ở:\n  - '
        + '\n  - '.join(str(c) for c in _candidates)
        + '\nCách khắc phục: mở thư mục được chia sẻ trong Drive (Shared with me), chuột phải '
          '"Add shortcut to Drive", chọn My Drive và giữ nguyên tên, rồi chạy lại cell này.'
    )

DRIVE_DATA = DRIVE_ROOT / 'data'
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_CHECKPOINTS = DRIVE_ROOT / 'checkpoints'

# 3. Bảo đảm cấu trúc thư mục (chỉ tạo thư mục còn thiếu, không xóa/ghi đè gì) -----------------
for _d in [DRIVE_DATA / 'plantvillage dataset', DRIVE_DATA / 'splits', DRIVE_DATA / 'samples',
           DRIVE_DATA / 'poisoned' / 'label_flipping', DRIVE_DATA / 'poisoned' / 'backdoor',
           *(DRIVE_RESULTS / p for p in PIPELINES), *(DRIVE_CHECKPOINTS / p for p in PIPELINES)]:
    _d.mkdir(parents=True, exist_ok=True)

# Kiểm tra quyền ghi (người chỉ có quyền xem sẽ không lưu được kết quả)
_probe = DRIVE_RESULTS / f'.write_test_{os.getpid()}'
try:
    _probe.write_text('ok')
    _probe.unlink()
except OSError as exc:
    raise PermissionError(
        f'Không ghi được vào {DRIVE_RESULTS}. Hãy nhờ chủ thư mục cấp quyền Editor. ({exc})') from exc


# 4. Lấy code về đĩa local ------------------------------------------------------------------
def _sh(cmd, secret=None):
    shown = ' '.join(cmd)
    if secret:
        shown = shown.replace(secret, '***')
    print('$', shown)
    done = subprocess.run(cmd, text=True, capture_output=True)
    out = (done.stdout + done.stderr).strip()
    if secret:
        out = out.replace(secret, '***')
    if out:
        print(out)
    if done.returncode != 0:
        raise RuntimeError(f'Lệnh thất bại (exit {done.returncode}).')


if REPO_URL:
    _url, _token = REPO_URL, None
    if _url.startswith('https://') and 'github.com' in _url:
        try:                                    # repo private: lưu token trong Colab Secrets với tên GITHUB_TOKEN
            from google.colab import userdata
            _token = userdata.get('GITHUB_TOKEN')
        except Exception:
            _token = None
        if _token:
            _url = _url.replace('https://', f'https://{_token}@', 1)
    if (LOCAL_PROJECT / '.git').is_dir():
        _sh(['git', '-C', str(LOCAL_PROJECT), 'pull', '--ff-only'], secret=_token)
    else:
        _sh(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, _url, str(LOCAL_PROJECT)], secret=_token)
    CODE_SOURCE = f'GitHub {REPO_URL} ({REPO_BRANCH})'
else:
    _src = DRIVE_ROOT / 'repo'
    if not _src.is_dir():
        raise FileNotFoundError(
            f'Chưa có {_src}. Hãy upload code vào thư mục repo/ trong Drive dùng chung '
            '(chỉ code, không kèm data/results/checkpoints), hoặc điền REPO_URL.')
    _skip_anywhere = {'.git', '__pycache__', '.ipynb_checkpoints', 'results', 'checkpoints'}
    _skip_top = {'data', 'outputs', 'models'}

    def _ignore(directory, names):
        skip = {n for n in names if n in _skip_anywhere or n.endswith(('.pth', '.pt', '.onnx'))}
        if Path(directory) == _src:
            skip |= {n for n in names if n in _skip_top}
        return skip

    shutil.copytree(_src, LOCAL_PROJECT, dirs_exist_ok=True, ignore=_ignore)
    CODE_SOURCE = f'Drive: {_src}'

PROJECT_ROOT = LOCAL_PROJECT
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 5. Nối các thư mục dữ liệu/kết quả/checkpoint của project vào Drive (symlink) ---------------
#    Code không cần sửa: đường dẫn tương đối trong project tự trỏ vào Drive.
LINKS = {
    'data/plantvillage dataset': DRIVE_DATA / 'plantvillage dataset',
    'data/splits': DRIVE_DATA / 'splits',
    'data/poisoned': DRIVE_DATA / 'poisoned',
    'data/samples': DRIVE_DATA / 'samples',
    'outputs': DRIVE_RESULTS,
    'models/checkpoints': DRIVE_CHECKPOINTS,
}
for _p in PIPELINES:
    LINKS[f'experiments/{_p}/results'] = DRIVE_RESULTS / _p
    LINKS[f'experiments/{_p}/checkpoints'] = DRIVE_CHECKPOINTS / _p


def _link(rel, target):
    path = PROJECT_ROOT / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.is_symlink():
        if Path(os.readlink(path)) == target:
            return 'OK'
        path.unlink()
    elif path.exists():
        if path.is_dir() and not any(path.iterdir()):
            path.rmdir()
        else:
            return 'BỎ QUA: đã có dữ liệu thật trong thư mục local, không ghi đè'
    os.symlink(target, path, target_is_directory=True)
    return 'OK'


_link_status = {rel: _link(rel, tgt) for rel, tgt in LINKS.items()}

# 6. Biến môi trường cho code mới (attack/defense) ---------------------------------------------
os.environ.update({
    'DP_DRIVE_ROOT': str(DRIVE_ROOT),
    'DP_DATA_ROOT': str(DRIVE_DATA),
    'DP_RESULTS_ROOT': str(DRIVE_RESULTS),
    'DP_CHECKPOINTS_ROOT': str(DRIVE_CHECKPOINTS),
})


def finish_session():
    """Gọi ở cuối phiên (hoặc trước khi bạn khác tiếp tục) để chắc chắn dữ liệu đã đồng bộ lên Drive."""
    if IN_COLAB:
        from google.colab import drive as _drive
        _drive.flush_and_unmount()
        print('Đã đồng bộ và ngắt kết nối Drive.')
    else:
        print('Không ở trong Colab: không cần đồng bộ.')


# 7. Tóm tắt -----------------------------------------------------------------------------------
print(f'Drive root : {DRIVE_ROOT}')
print(f'Code       : {PROJECT_ROOT}  <-  {CODE_SOURCE}')
print('Liên kết (project -> Drive):')
for _rel, _st in _link_status.items():
    print(f'  {_rel:<40} {_st}')
_raw = DRIVE_DATA / 'plantvillage dataset' / 'color'
_has = {
    'ảnh gốc (data/plantvillage dataset/color)': _raw.is_dir() and any(_raw.iterdir()),
    'manifest_train.csv': (DRIVE_DATA / 'splits' / 'manifest_train.csv').is_file(),
    'manifest_test.csv': (DRIVE_DATA / 'splits' / 'manifest_test.csv').is_file(),
    'class_to_idx.json': (DRIVE_DATA / 'splits' / 'class_to_idx.json').is_file(),
}
print('Dữ liệu dùng chung:')
for _name, _ok in _has.items():
    print(f'  {"CÓ   " if _ok else "THIẾU"} {_name}')
print('Quyền ghi: OK')

Mounted at /content/drive
$ git clone --depth 1 --branch main https://github.com/qtrong0205/data-poisoning-research /content/data-poisoning-plantvillage
Cloning into '/content/data-poisoning-plantvillage'...
Drive root : /content/drive/MyDrive/data-poisoning-shared
Code       : /content/data-poisoning-plantvillage  <-  GitHub https://github.com/qtrong0205/data-poisoning-research (main)
Liên kết (project -> Drive):
  data/plantvillage dataset                OK
  data/splits                              OK
  data/poisoned                            OK
  data/samples                             OK
  outputs                                  OK
  models/checkpoints                       OK
  experiments/baseline/results             OK
  experiments/baseline/checkpoints         OK
  experiments/label_flipping/results       OK
  experiments/label_flipping/checkpoints   OK
  experiments/backdoor/results             OK
  experiments/backdoor/checkpoints         OK
  experiments/defenses/results

In [ ]:
# Kiểm tra các thư mục kết quả/checkpoint đã thật sự nối vào Drive (nếu không, kết quả sẽ MẤT khi hết phiên)
_bad = {rel: st for rel, st in _link_status.items() if st != 'OK'}
if _bad:
    raise RuntimeError(
        'Các thư mục sau chưa được nối vào Drive:\n'
        + '\n'.join(f'  {rel}: {st}' for rel, st in _bad.items())
        + '\nNguyên nhân thường gặp: thư mục đó vẫn còn file được commit trong repo (hãy git rm --cached '
          'và thêm vào .gitignore). Không chạy training khi lỗi này chưa được xử lý.')
print(f'Đã nối {len(_link_status)} thư mục của project vào Drive.')

Đã nối 14 thư mục của project vào Drive.


In [ ]:
# Cài dependency, bỏ qua torch/torchvision/CUDA để không ghi đè PyTorch sẵn có của Colab
import re

SKIP_PACKAGES = {'torch', 'torchvision', 'torchaudio', 'triton'}
packages = []
for line in (PROJECT_ROOT / 'requirements.txt').read_text(encoding='utf-8').splitlines():
    item = line.strip()
    if not item or item.startswith(('#', '-')):
        continue
    name = re.split(r'[=<>!~;\[\s@]', item, maxsplit=1)[0].lower().replace('_', '-')
    if name in SKIP_PACKAGES or name.startswith('nvidia-'):
        continue
    packages.append(item)
if packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print(f'Đã cài {len(packages)} package.')

Đã cài 13 package.


In [ ]:
import torch
import torchvision
import yaml
import sklearn
import matplotlib
import importlib
import numpy as np
import pandas as pd
import PIL
from tqdm.auto import tqdm

print("Dependencies imported successfully!")


Dependencies imported successfully!


In [ ]:
# Kiểm tra dữ liệu dùng chung. Notebook KHÔNG tự tải dataset hay tạo split:
# việc đó do người quản lý làm một lần (nhiều người cùng ghi vào Drive dùng chung sẽ gây lỗi).
import pandas as pd

split_dir = PROJECT_ROOT / 'data/splits'
for _name in ('manifest_train.csv', 'manifest_test.csv', 'class_to_idx.json'):
    if not (split_dir / _name).is_file():
        raise FileNotFoundError(f'Thiếu {split_dir / _name}. Hãy báo người quản lý chuẩn bị dữ liệu.')

# So số ảnh trong manifest với số file thật của từng lớp (nhanh, chỉ liệt kê thư mục)
_problems = []
for _split in ('train', 'test'):
    _df = pd.read_csv(split_dir / f'manifest_{_split}.csv')
    for _cls, _n in _df.groupby('class_name').size().items():
        _folder = split_dir / _split / _cls
        _have = sum(1 for e in os.scandir(_folder) if e.is_file()) if _folder.is_dir() else 0
        if _have < _n:
            _problems.append({'split': _split, 'class': _cls, 'manifest': int(_n), 'on_drive': _have})
if _problems:
    display(pd.DataFrame(_problems))
    raise FileNotFoundError(f'{len(_problems)} lớp thiếu ảnh trong data/splits (bảng trên). Báo người quản lý bổ sung trước khi train.')
print('data/splits đầy đủ so với manifest.')

data/splits đầy đủ so với manifest.


In [ ]:
# Hàm chạy lệnh con và in log RA CELL NGAY KHI có (kể cả thanh tiến trình tqdm dùng ký tự \r).
# subprocess.run(...) trần không hiện log trong cell Colab, nên cell trông như bị "đóng băng".
import codecs
import os
import subprocess
import sys


def run_streaming(cmd, cwd=None):
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    decoder = codecs.getincrementaldecoder('utf-8')('replace')   # không làm hỏng chữ có dấu bị cắt giữa chunk
    try:
        while True:
            chunk = proc.stdout.read1(4096)       # trả về ngay khi có dữ liệu, không chờ xuống dòng
            if not chunk:
                break
            sys.stdout.write(decoder.decode(chunk))
            sys.stdout.flush()
        sys.stdout.write(decoder.decode(b'', final=True))
    except BaseException:                         # bấm Stop trên Colab: dừng luôn tiến trình train, tránh chạy mồ côi
        proc.kill()
        proc.wait()
        raise
    code = proc.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)
    return code


# ---------------------------------------------------------------------------------
# Đọc hàng chục nghìn ảnh nhỏ trực tiếp từ Drive rất chậm và có thể đứng rất lâu (nhất là lần đọc đầu).
# Chép ảnh về đĩa local của Colab (song song) rồi train từ bản local. Kết quả/checkpoint vẫn ghi vào Drive.
# ---------------------------------------------------------------------------------
import shutil
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd

USE_LOCAL_SPLITS = True                       # False: đọc thẳng từ Drive (chậm)
LOCAL_SPLITS = Path('/content/splits_local')


def prepare_local_splits(only_subset, workers=32):
    if not USE_LOCAL_SPLITS:
        print('USE_LOCAL_SPLITS=False: đọc ảnh trực tiếp từ Drive.', flush=True)
        return
    src = DRIVE_DATA / 'splits'
    train = pd.read_csv(src / 'manifest_train.csv')
    if only_subset:
        train = train[train.in_subset == 1]
    test = pd.read_csv(src / 'manifest_test.csv')
    rels = list(train.rel_path) + list(test.rel_path)
    todo = [r for r in rels if not (LOCAL_SPLITS / r).is_file()]
    print(f'Chép {len(todo):,}/{len(rels):,} ảnh từ Drive về {LOCAL_SPLITS} (đã có sẵn {len(rels) - len(todo):,}).', flush=True)
    LOCAL_SPLITS.mkdir(parents=True, exist_ok=True)
    for name in ('manifest_train.csv', 'manifest_test.csv', 'class_to_idx.json'):
        shutil.copy2(src / name, LOCAL_SPLITS / name)

    def _copy(rel):
        dst = LOCAL_SPLITS / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        tmp = dst.with_name(dst.name + '.part')
        shutil.copyfile(src / rel, tmp)
        os.replace(tmp, dst)                  # ghi xong mới đổi tên: file dở dang không bị coi là đã chép

    t0, done = time.time(), 0
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for _ in pool.map(_copy, todo):
            done += 1
            if done % 2000 == 0:
                print(f'  đã chép {done:,}/{len(todo):,} ({done / (time.time() - t0):.0f} ảnh/giây)', flush=True)
    print(f'Xong {len(todo):,} ảnh trong {time.time() - t0:.0f}s.', flush=True)

    link = PROJECT_ROOT / 'data/splits'
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        raise RuntimeError(f'{link} là thư mục thật, không đổi sang bản local được.')
    link.symlink_to(LOCAL_SPLITS, target_is_directory=True)
    print(f'data/splits đang trỏ tới {LOCAL_SPLITS} (đĩa local).', flush=True)

In [ ]:
# Chạy 1 experiment label flipping: kiểm tra config -> bỏ qua run đã xong -> kiểm tra manifest poisoned
# -> chép ảnh về local -> train. Output nằm ở experiments/label_flipping/{results,checkpoints}/<E-LFx>/...
# nên đã được nối vào Drive sẵn bởi cell mount (không cần nối thêm).

# experiment.py KHÔNG tự tạo manifest poisoned. Nếu muốn notebook tự gọi prepare_data khi thiếu manifest,
# điền lệnh vào đây ({exp} sẽ được thay bằng ID experiment). Để None = chỉ báo lỗi rõ ràng nếu thiếu.
# Ví dụ: [sys.executable, '-m', 'src.attacks.label_flipping.prepare_data', '--experiment', '{exp}']
PREPARE_CMD = None


def run_label_flipping_experiment(experiment_id):
    lf = importlib.reload(importlib.import_module('experiments.label_flipping.experiment'))   # dùng lại hàm của experiment.py
    attack_cfg, dataset_cfg, baseline_cfg = lf.load_config()

    try:
        settings = lf._settings(attack_cfg, experiment_id)
    except ValueError as exc:
        print(f'[{experiment_id}] CHƯA CHẠY ĐƯỢC: {exc}', flush=True)
        return

    seeds = settings.get('seeds', [settings.get('seed', 42)])
    runs = [(float(rate), int(seed)) for rate in settings['poisoning_rates'] for seed in seeds]
    out_root = lf.PROJECT_ROOT / 'experiments' / 'label_flipping'

    pending = []
    for rate, seed in runs:
        rate_name = lf._rate_name(rate)
        run_root = out_root / 'results' / experiment_id / rate_name / f'seed_{seed}'
        ckpt = out_root / 'checkpoints' / experiment_id / rate_name / f'seed_{seed}' / 'final_model.pth'
        if lf._completed_run(run_root, ckpt, experiment_id, rate, seed) is None:
            pending.append((rate, seed))
    print(f'[{experiment_id}] strategy={settings["strategy"]}, scope={settings["dataset_scope"]}: '
          f'{len(runs)} run, đã xong {len(runs) - len(pending)}, cần chạy {len(pending)}.', flush=True)
    if not pending:
        print(f'[{experiment_id}] Đã có kết quả hợp lệ trên Drive, bỏ qua training.', flush=True)
        return

    # Kiểm tra manifest poisoned trước khi chép ảnh (để lỗi thì lỗi sớm)
    manifest_root = PROJECT_ROOT / attack_cfg['data']['poisoned_manifest_root'] / experiment_id

    def _missing_rates():
        return [r for r in sorted({rate for rate, _ in pending})
                if not (manifest_root / lf._rate_name(r) / 'manifest_train.csv').is_file()]

    missing = _missing_rates()
    if missing and PREPARE_CMD:
        print(f'[{experiment_id}] Thiếu manifest cho rate {missing}, chạy prepare_data...', flush=True)
        run_streaming([c.format(exp=experiment_id) for c in PREPARE_CMD], cwd=PROJECT_ROOT)
        missing = _missing_rates()
    if missing:
        raise FileNotFoundError(
            f'[{experiment_id}] Thiếu manifest poisoned cho rate {missing} trong {manifest_root}. '
            'Hãy chạy prepare_data.py cho experiment này trước (hoặc điền PREPARE_CMD ở cell trên).')

    prepare_local_splits(only_subset=(settings['dataset_scope'] == 'subset'))
    print(f'[{experiment_id}] launching training subprocess...', flush=True)
    run_streaming(
        [sys.executable, '-u', '-m', 'experiments.label_flipping.experiment', '--experiment', experiment_id],
        cwd=PROJECT_ROOT,
    )
    print(f'[{experiment_id}] subprocess completed.', flush=True)
    print(f'Results are stored in: {DRIVE_RESULTS / "label_flipping" / experiment_id}')
    print(f'Checkpoints are stored in: {DRIVE_CHECKPOINTS / "label_flipping" / experiment_id}')


In [ ]:
# E-LF1: random label flipping - rate sweep
%cd /content/data-poisoning-plantvillage
run_label_flipping_experiment('E-LF1')


/content
[E-LF1] strategy=random, scope=subset: 7 run, đã xong 7, cần chạy 0.
[E-LF1] Đã có kết quả hợp lệ trên Drive, bỏ qua training.


In [ ]:
# E-LF2: targeted label flipping
%cd /content/data-poisoning-plantvillage
run_label_flipping_experiment('E-LF2')


/content
[E-LF2] strategy=targeted, scope=subset: 3 run, đã xong 3, cần chạy 0.
[E-LF2] Đã có kết quả hợp lệ trên Drive, bỏ qua training.


In [ ]:
# E-LF3: confirmatory experiments (full data, 3 seeds)
%cd /content/data-poisoning-plantvillage
run_label_flipping_experiment('E-LF3')


/content
[E-LF3] strategy=random, scope=full: 9 run, đã xong 0, cần chạy 9.
Chép 54,305/54,305 ảnh từ Drive về /content/splits_local (đã có sẵn 0).
  đã chép 2,000/54,305 (107 ảnh/giây)
  đã chép 4,000/54,305 (114 ảnh/giây)
  đã chép 6,000/54,305 (116 ảnh/giây)
  đã chép 8,000/54,305 (117 ảnh/giây)
  đã chép 10,000/54,305 (119 ảnh/giây)
  đã chép 12,000/54,305 (120 ảnh/giây)
  đã chép 14,000/54,305 (121 ảnh/giây)
  đã chép 16,000/54,305 (122 ảnh/giây)
  đã chép 18,000/54,305 (123 ảnh/giây)
  đã chép 20,000/54,305 (123 ảnh/giây)
  đã chép 22,000/54,305 (124 ảnh/giây)
  đã chép 24,000/54,305 (123 ảnh/giây)
  đã chép 26,000/54,305 (124 ảnh/giây)
  đã chép 28,000/54,305 (124 ảnh/giây)
  đã chép 30,000/54,305 (124 ảnh/giây)
  đã chép 32,000/54,305 (124 ảnh/giây)
  đã chép 34,000/54,305 (124 ảnh/giây)
  đã chép 36,000/54,305 (124 ảnh/giây)
  đã chép 38,000/54,305 (125 ảnh/giây)
  đã chép 40,000/54,305 (125 ảnh/giây)
  đã chép 42,000/54,305 (125 ảnh/giây)
  đã chép 44,000/54,305 (125 ảnh/giây

In [ ]:
# Xem kết quả label flipping đã có trên Drive
def _scalars(prefix, d):
    return {f'{prefix}_{k}': v for k, v in (d or {}).items() if isinstance(v, (int, float)) and not isinstance(v, bool)}


_rows = []
for _p in sorted((DRIVE_RESULTS / 'label_flipping').glob('*/rate_*/seed_*/metrics.json')):
    _m = json.loads(_p.read_text(encoding='utf-8'))
    _row = {'experiment': _m['experiment_name'], 'strategy': _m['strategy'], 'rate': _m['poisoning_rate'],
            'seed': _m['run_seed'], 'epochs': _m['epochs'], 'train': _m['train_samples']}
    _row.update(_scalars('clean', _m.get('evaluation')))
    _row.update(_scalars('attack', _m.get('attack')))
    _rows.append(_row)

if _rows:
    _df = pd.DataFrame(_rows).sort_values(['experiment', 'rate', 'seed']).reset_index(drop=True)
    display(_df)
    _metric_cols = [c for c in _df.columns if c.startswith(('clean_', 'attack_'))]
    display(_df.groupby(['experiment', 'strategy', 'rate'])[_metric_cols].agg(['mean', 'std']))
else:
    print('Chưa có kết quả nào.')


,experiment,strategy,rate,seed,epochs,train,clean_accuracy,clean_precision,clean_recall,clean_f1_score
0,E-LF1,random,0.01,42,3,13028,0.930305,0.905019,0.912083,0.901524
1,E-LF1,random,0.02,42,3,13028,0.926535,0.911791,0.903491,0.898854
2,E-LF1,random,0.05,42,3,13028,0.902630,0.890440,0.865833,0.862389
3,E-LF1,random,0.10,42,3,13028,0.888194,0.869399,0.843619,0.833720
4,E-LF1,random,0.20,42,3,13028,0.853623,0.839403,0.785556,0.789338
5,E-LF1,random,0.30,42,3,13028,0.783560,0.798178,0.722762,0.726815
6,E-LF1,random,0.40,42,3,13028,0.707797,0.746201,0.628262,0.619593
7,E-LF2,targeted,0.05,42,3,13028,0.932696,0.917294,0.911185,0.908484
8,E-LF2,targeted,0.10,42,3,13028,0.930305,0.913223,0.907524,0.904491
9,E-LF2,targeted,0.20,42,3,13028,0.923685,0.910638,0.899390,0.891780


clean_accuracy           clean_precision            \
                                   mean       std            mean       std   
experiment strategy rate                                                      
E-LF1      random   0.01       0.930305       NaN        0.905019       NaN   
                    0.02       0.926535       NaN        0.911791       NaN   
                    0.05       0.902630       NaN        0.890440       NaN   
                    0.10       0.888194       NaN        0.869399       NaN   
                    0.20       0.853623       NaN        0.839403       NaN   
                    0.30       0.783560       NaN        0.798178       NaN   
                    0.40       0.707797       NaN        0.746201       NaN   
E-LF2      targeted 0.05       0.932696       NaN        0.917294       NaN   
                    0.10       0.930305       NaN        0.913223       NaN   
                    0.20       0.923685       NaN        0.910638       NaN   
E-LF3      random   0.05       0.921601  0.007410        0.908577  0.009572   
                    0.10       0.911058  0.001894        0.892011  0.008198   
                    0.20       0.869161  0.001460        0.855764  0.004131   

                         clean_recall           clean_f1_score            
                                 mean       std           mean       std  
experiment strategy rate                                                  
E-LF1      random   0.01     0.912083       NaN       0.901524       NaN  
                    0.02     0.903491       NaN       0.898854       NaN  
                    0.05     0.865833       NaN       0.862389       NaN  
                    0.10     0.843619       NaN       0.833720       NaN  
                    0.20     0.785556       NaN       0.789338       NaN  
                    0.30     0.722762       NaN       0.726815       NaN  
                    0.40     0.628262       NaN       0.619593       NaN  
E-LF2      targeted 0.05     0.911185       NaN       0.908484       NaN  
                    0.10     0.907524       NaN       0.904491       NaN  
                    0.20     0.899390       NaN       0.891780       NaN  
E-LF3      random   0.05     0.901159  0.007250       0.900277  0.009997  
                    0.10     0.888407  0.003048       0.884146  0.004746  
                    0.20     0.840087  0.003431       0.834319  0.001970

In [ ]:
# Chạy khi kết thúc phiên (sau khi đã xem xong kết quả). Sau lệnh này Drive bị ngắt:
# muốn chạy tiếp phải chạy lại cell mount (hoặc khởi động lại runtime).
finish_session()

Đã đồng bộ và ngắt kết nối Drive.
